# GÓC NHÌN MỚI: `head + đoạn` — thí nghiệm CÓ ĐỐI CHỨNG

## Vì sao
`enrich` 23/08 bị đọc sai. Ba nhánh đều dưới mốc sản xuất **vì chúng dùng `pick_chunks(k=1)`**,
không phải vì thêm chữ là xấu. So giữa ba nhánh với nhau (cùng câu, cùng ứng viên, cùng model,
chỉ khác văn bản đầu vào):

| | top-1 | Δ | McNemar |
|---|---|---|---|
| `front` (head đặt TRƯỚC) vs `plain` | 0.5533 vs 0.5133 | **+4,00** | 27–15, p=0.088 |
| **`front` vs `back` (head đặt SAU)** | 0.5533 vs 0.5000 | **+5,33** | 29–13, **p=0.0195** |
| `plain` vs `back` | 0.5133 vs 0.5000 | +1,33 | p=0.503 |

`back` có ĐÚNG BẰNG lượng chữ mà không ăn gì ⇒ không phải "nhiều chữ hơn", mà là **phần đầu
văn bản, đặt trước**. Nó chứa tên + phạm vi điều chỉnh — thứ phân biệt hai văn bản gần giống nhau,
và 67% câu sai có gold ở hạng 2–3.

**Tín hiệu này CHƯA có trong pipeline:** `ce` chấm 3 đoạn của D; `ce_deep` chấm K đoạn do
`pick_chunks` chọn **theo BM25 với câu hỏi** — phần mở đầu trùng từ với câu hỏi rất ít nên
gần như không bao giờ được chọn.

## ĐỐI CHỨNG — chỗ này là điểm mấu chốt của thiết kế
Chấm **HAI** góc nhìn mới trên cùng một tập đoạn:
- `ce_head`  = `head[:260] + "\n\n" + đoạn`
- `ce_plain` = `đoạn` (KHÔNG head)

Thiếu `ce_plain` thì không phân biệt được "head có ích" với "thêm góc nhìn nào cũng có ích".
Đây đúng là lỗi làm hỏng kết luận `enrich` lần trước.

## Ngưỡng đặt TRƯỚC khi chạy — mốc `max(ce, ce_deep)` = 0.7100

| `max(ce,ce_deep,ce_head)` | kết luận |
|---|---|
| **≥ 0,73** và hơn `ce_plain` | head ăn thật → dựng bài public, nộp |
| 0,715 – 0,73 | có dấu hiệu → nộp 1 bài thăm dò LB (n=1000 phân giải tốt hơn) |
| < 0,715 | đóng hướng, đừng quay lại |

Nếu `ce_head` ≈ `ce_plain` thì **head vô dụng**, cái ăn được chỉ là góc nhìn thứ ba — ghi rõ vậy.

## Chi phí
300 câu × 5 văn bản × 3 đoạn × 2 nhánh = **9.000 cặp ≈ 12 phút GPU** @12,9 cặp/s.
Chạy độc lập, KHÔNG đụng lượt `fusion_dev1000` đang chạy.

## Upload lên Kaggle
`dev_300_locked.json` · `scores_dev300_fusion_M20_K20.json` · `selected-contexts/` ·
`deep_chunk.py` + `rerank.py`

**Không phải sửa đường dẫn** — ô 1 tự dò mọi thứ dưới `/kaggle/input` và in ra chỗ nó tìm thấy.

In [ ]:
import os, sys, json, time, gc
import numpy as np, torch

# ===== TỰ DÒ MỌI ĐƯỜNG DẪN DƯỚI /kaggle/input — không phải sửa tay =====
ROOT = "/kaggle/input"

def find_file(name):
    for r, _, fs in os.walk(ROOT):
        if name in fs: return os.path.join(r, name)
    raise FileNotFoundError(f"KHÔNG THẤY {name} dưới {ROOT} — kiểm tra đã Add Data chưa")

def find_ctx():
    for r, _, fs in os.walk(ROOT):
        for f in fs:
            if f.startswith("context_") and f.endswith(".json"): return r
    raise FileNotFoundError(f"KHÔNG THẤY thư mục chứa context_*.json dưới {ROOT}")

DEV    = find_file("dev_300_locked.json")
SCORES = find_file("scores_dev300_fusion_M20_K20.json")
UTIL   = os.path.dirname(find_file("deep_chunk.py"))
CTX    = find_ctx()
for n, v in [("DEV", DEV), ("SCORES", SCORES), ("UTIL", UTIL), ("CTX", CTX)]:
    print(f"  {n:<7} {v}")
print(f"  kho có {sum(1 for f in os.listdir(CTX) if f.startswith('context_')):,} văn bản\n")

OUT = "/kaggle/working/head_view_dev300.json"
TOP_DOCS, K_VIEW = 5, 3
HEAD_CHARS, EXC_CHARS, SEP = 260, 900, "\n\n"     # y hệt enrich_run -> so sánh được

sys.path.insert(0, UTIL)
import deep_chunk as DC
from rerank import load_reranker
DC.MERGE_CHARS = 1800                              # cùng cấu hình bài chốt

dev = json.load(open(DEV, encoding="utf-8"))
S   = json.load(open(SCORES, encoding="utf-8"))
gold = {q: {str(x) for x in v["answer"]} for q, v in dev.items()}
Q    = list(gold)

mx    = lambda v: max(v["ce"], v.get("ce_deep", -9e9))
order = {q: [d for d, _ in sorted(S[q].items(), key=lambda kv: -mx(kv[1]))] for q in Q}
acc   = lambda pick: np.mean([pick[q] in gold[q] for q in Q])

base = acc({q: order[q][0] for q in Q})
print(f"mốc dựng lại = {base:.4f}   (phải là 0.7100)")
assert abs(base - 0.7100) < 1e-6, "SAI FILE ĐIỂM — dừng, đừng đọc Δ nào phía sau"
print(f"trần top-{TOP_DOCS} = {np.mean([bool(gold[q] & set(order[q][:TOP_DOCS])) for q in Q]):.4f}")
print(f"{len(Q)} câu · {TOP_DOCS} văn bản × {K_VIEW} đoạn × 2 nhánh = {len(Q)*TOP_DOCS*K_VIEW*2:,} cặp")

# kiểm read_passage đọc được thật, TRƯỚC khi nạp model
_t = DC.read_passage(CTX, order[Q[0]][0])
assert len(_t) > 50, "read_passage trả về rỗng — CTX sai thư mục"
print(f"read_passage OK ({len(_t):,} ký tự ở văn bản thử)")

In [ ]:
# ===== Chấm HAI nhánh trên CÙNG tập đoạn. Lưu dần, chạy lại là nối tiếp. =====
model = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda", max_length=1024)
res = json.load(open(OUT, encoding="utf-8")) if os.path.isfile(OUT) else {}
print(f"đã có {len(res)}/{len(Q)} câu")

_hd, t0 = {}, time.time()
def head_of(d):
    if d not in _hd: _hd[d] = DC.read_passage(CTX, d)[:HEAD_CHARS].strip()
    return _hd[d]

todo = [q for q in Q if q not in res]
for i, q in enumerate(todo, 1):
    qt = dev[q]["question"]
    pairs, owner = [], []
    for d in order[q][:TOP_DOCS]:
        h = head_of(d)
        for c in DC.pick_chunks(qt, CTX, d, k=K_VIEW):
            c = c[:EXC_CHARS]
            pairs += [[qt, (h + SEP + c) if h else c], [qt, c]]   # head / plain, CÙNG đoạn
            owner += [(d, "head"), (d, "plain")]
    sc = model.predict(pairs, batch_size=32, show_progress_bar=False)
    per = {}
    for (d, arm), v in zip(owner, sc):
        per.setdefault(d, {}).setdefault(arm, []).append(float(v))
    res[q] = per

    if i % 50 == 0 or i == len(todo):
        json.dump(res, open(OUT, "w", encoding="utf-8"), ensure_ascii=False)
        el = time.time() - t0
        print(f"  {i}/{len(todo)} câu · {el/60:.1f} phút · còn ~{el/i*(len(todo)-i)/60:.1f} phút", flush=True)
        gc.collect(); torch.cuda.empty_cache()

json.dump(res, open(OUT, "w", encoding="utf-8"), ensure_ascii=False)
print(f"\nXONG -> {OUT}   (TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN — quy tắc 2)")

In [ ]:
# ===== Đo, có đối chứng, có McNemar =====
from math import comb
res = json.load(open(OUT, encoding="utf-8"))
assert len(res) == len(Q), f"mới chấm {len(res)}/{len(Q)} câu"

def pick(use_old=True, arm=None):
    out = {}
    for q in Q:
        best, bs = None, -9e9
        for d in order[q][:TOP_DOCS]:
            s = [mx(S[q][d])] if use_old else []
            if arm: s += res[q].get(d, {}).get(arm, [])
            m = max(s) if s else -9e9
            if m > bs: bs, best = m, d
        out[q] = best
    return out

def mcnemar(a, b):
    A = np.array([a[q] in gold[q] for q in Q]); B = np.array([b[q] in gold[q] for q in Q])
    w = int((A & ~B).sum()); l = int((B & ~A).sum()); n = w + l
    p = (sum(comb(n, i) for i in range(max(w, l), n + 1)) / 2**n * 2) if n else 1.0
    return A.mean(), w, l, min(p, 1.0)

REF = pick(True, None)                       # = max(ce, ce_deep) trong top-5
rows = [("MỐC  max(ce,ce_deep)",      REF),
        ("chỉ ce_head (không cũ)",    pick(False, "head")),
        ("chỉ ce_plain (không cũ)",   pick(False, "plain")),
        ("max(cũ, ce_plain)  ĐỐI CHỨNG", pick(True, "plain")),
        ("max(cũ, ce_head)   THẬT",   pick(True, "head"))]

print(f"{'cấu hình':<32}{'top-1':>8}{'Δ vs mốc':>10}   McNemar vs mốc")
print("-" * 74)
for name, p in rows:
    a, w, l, pv = mcnemar(p, REF)
    tail = "" if name.startswith("MỐC") else f"   thắng {w} thua {l}  p={pv:.4f}"
    print(f"{name:<32}{a:>8.4f}{a-base:>+10.4f}{tail}")

head_v, plain_v = pick(True, "head"), pick(True, "plain")
a, w, l, pv = mcnemar(head_v, plain_v)
print(f"\nHEAD có ích hơn CHỈ-THÊM-GÓC-NHÌN?  {a:.4f} vs {mcnemar(plain_v,REF)[0]:.4f}"
      f"  thắng {w} thua {l}  p={pv:.4f}")

got = mcnemar(head_v, REF)[0]
print("\n" + "=" * 66)
if   got >= 0.73:  print(f"{got:.4f} -> ĂN THẬT. Dựng bài public (1000 câu, ~20 phút GPU), nộp.")
elif got >= 0.715: print(f"{got:.4f} -> CÓ DẤU HIỆU. Nộp 1 bài thăm dò LB (n=1000 phân giải tốt hơn dev300).")
else:              print(f"{got:.4f} -> DƯỚI NGƯỠNG. Đóng hướng, đừng quay lại.")
print("=" * 66)